In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Deploy ADK Agent in AI Engine

This notebook provides a step-by-step to deploy an Agent Created using Agent Development Kit on Agent Engine (ReasonEngine on Vertex AI)

**Important**: This notebook consider that the Agent was built with ADK and the agent files are inside an agent folder and the dependencies in a file config.yaml

Folders structure (example): 
```
parent_folder/
    agent_folder/
        __init__.py
        agent.py
        config.yaml
        .env
    deploy_agent_engine.ipynb
```

config.yaml (example): 
```yaml
agent_name: 'agent_name'
agent_display_name: 'Agent Name'
agent_description: 'Useful agent to help users'

deploy:
  dependencies: ['google-cloud-aiplatform[agent_engines]', 'google-adk', 'cloudpickle', 'pydantic']
```


### Setup and Config

In [19]:
# Checking the google-adk and google-cloud-aiplatform versions
!pip freeze | grep google-adk
!pip freeze | grep google-cloud-aiplatform

google-adk==1.18.0
google-cloud-aiplatform==1.126.1


In [ ]:
# Authentication on gcloud (if necessary)
# !gcloud auth application-default login

In [20]:
# Basic Libraries
import os 
import vertexai
import yaml

# AI Engine on Vertex AI 
from vertexai import agent_engines

# Library for AI Engine with ADK
from vertexai.preview import reasoning_engines

# Just to view JSON response formatted
import json
from IPython.display import display,Markdown,JSON

# To load envvars dict from .env file
from dotenv import dotenv_values

In [21]:
# Load Agent Config
AGENT_DIR = "adk_bq_agent"

# Load environment variables from .env file from agent Directory 
from dotenv import load_dotenv
env_file = f'./{AGENT_DIR}/.env'
load_dotenv(env_file)

# Load config from agents (Params and dependencies for deploy)
with open(f'./{AGENT_DIR}/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# For Vertex AI SDK 
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION")
BUCKET = os.environ.get("GOOGLE_CLOUD_BUCKET")

### Instantiate Agent from Directory

In [22]:
# Importing Agent Module from AGENT_DIR folder
import importlib
agent_module = importlib.import_module(f"{AGENT_DIR.replace('/','.')}.agent")

# Instantiate the Assistant as an ADK App 
adk_agent = reasoning_engines.AdkApp(
    agent=agent_module.root_agent,
    enable_tracing=True
)

### Running Agent Local (Optional)

In [23]:
# Run a simple query
for event in adk_agent.stream_query(
    user_id="user",
    message="Hi, how can you help me?",
):
    pass

# Formatted output
display(Markdown(f"```json\n{json.dumps(event, indent=2,ensure_ascii=False)}\n```"))

telemetry enabled but proceeding without GenAI instrumentation, because not all packages (i.e. opentelemetry-instrumentation-google-genai) have been installed


/Users/speca/Dev/google/adk_bq_agent/.venv/lib/python3.12/site-packages/vertexai/preview/reasoning_engines/templates/adk.py:798: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/Users/speca/Dev/google/adk_bq_agent/.venv/lib/python3.12/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
App name mismatch detected. The runner is configured with app name "default-app-name", but the root agent was loaded from "/Users/speca/Dev/google/adk_bq_agent/.venv/lib/python3.12/site-packages/google/adk/agents", which implies app name "agents".
Ap

```json
{
  "model_version": "gemini-2.5-flash",
  "content": {
    "parts": [
      {
        "text": "Hello! I can help you by writing and executing BigQuery SQL queries to retrieve information from the Moodle database.\n\nYou can ask me questions like:\n*   \"What are the names of all courses?\"\n*   \"How many users are enrolled in 'Quantum Gastronomy Fundamentals'?\"\n*   \"What is the average grade for 'Tarefa 1 - Q_GASTRO'?\"\n*   \"List all assignments due next month.\"\n*   \"Show me the completion status of modules for a specific user.\"\n\nJust tell me what information you're looking for!",
        "thought_signature": "CocBAePx_16kIICyyITL8Q9CBSstDajnzAQwyVnxTHn7er4ICxH-XPVXSs_dQWsweottX_fpVIdoiH40OeisKUwtLPzKL_cWPDL08T32E4hPEpAlVXY1c9OcZkZEC7tQ4TM6SOnEnLyhViEdh2Ct5UdgS7UKmGUl2V7N3L10F7SfbuwP0Ot3cYPn"
      }
    ],
    "role": "model"
  },
  "finish_reason": "STOP",
  "usage_metadata": {
    "candidates_token_count": 120,
    "candidates_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 120
      }
    ],
    "prompt_token_count": 9480,
    "prompt_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 9480
      }
    ],
    "thoughts_token_count": 23,
    "total_token_count": 9623,
    "traffic_type": "ON_DEMAND"
  },
  "avg_logprobs": -0.20125349362691244,
  "invocation_id": "e-d908b31e-aff6-460d-a4ca-6907831bd6e0",
  "author": "bigquery_agent",
  "actions": {
    "state_delta": {},
    "artifact_delta": {},
    "requested_auth_configs": {},
    "requested_tool_confirmations": {}
  },
  "id": "37d298da-c1c2-4664-9d66-30c77c0f5bc3",
  "timestamp": 1763741138.829051
}
```

### Deploy on Agent Engine

In [24]:
# Instantiate Vertex AI
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET,
)

In [25]:
## Retrieve all existent Agent Engine on your project
for agent in agent_engines.list():
    print(f"============================ \nAgent: {agent.display_name}\nResoruce Name: {agent.resource_name}\nCreated/updated at: {agent.update_time} \n\n" )

Agent: ADK BQ Agent OAuth
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/7451852096321617920
Created/updated at: 2025-11-21 15:34:54.761510+00:00 


Agent: Agente Recursos Humanos
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/8886459683871653888
Created/updated at: 2025-11-13 20:31:33.996791+00:00 


Agent: ADK VAIS Search Agent
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/5370837224755560448
Created/updated at: 2025-11-11 18:59:13.814593+00:00 


Agent: ADK OAuth Agent
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/1907964935493648384
Created/updated at: 2025-10-28 17:50:34.549372+00:00 




In [26]:
# Read Requirements for Agent from config file
# Usually this ['google-cloud-aiplatform[agent_engines]', 'google-adk', 'cloudpickle'] plus the packages that agent needs
requirements = config['deploy']['dependencies']
requirements

['google-cloud-aiplatform[agent_engines,adk]',
 'google-adk',
 'cloudpickle',
 'pydantic',
 'python-dotenv',
 'pyyaml',
 'google-cloud-bigquery']

In [27]:
# Extra packages from agent folder (This is all .py files inside Agent Directory)
extra_packages = [AGENT_DIR]
extra_packages

['adk_bq_agent']

In [28]:
# Load Variables on env_vars dict to be used when creating the Agent
env_vars = dotenv_values(dotenv_path=env_file)

# Remove GCP variables (this variables already are defined in Agent Engine and are reserved)
keys_to_remove = [
    "GOOGLE_GENAI_USE_VERTEXAI",
    "GOOGLE_CLOUD_PROJECT",
    "GOOGLE_CLOUD_LOCATION",
    "GOOGLE_CLOUD_BUCKET"
]

for key in keys_to_remove:
    env_vars.pop(key, None)


In [29]:
#UPDATE REASONING ENGINE WITH ALREADY EXIST

resource_name="projects/267339081837/locations/us-central1/reasoningEngines/7451852096321617920"

# Deploy the Agent on AI Engine (This takes a few minutes)
remote_agent = agent_engines.update(
    resource_name,
    agent_engine = adk_agent,             # The Agent instantiated as ADK agent
    requirements=requirements,            # Requirements file
    extra_packages=extra_packages,        # Extra packages
    display_name= config['agent_display_name'],    # Display name  
    description= config['agent_description'],     # Description
    env_vars=env_vars                     # Env Vars dict
)

Identified the following requirements: {'google-cloud-aiplatform': '1.126.1', 'pydantic': '2.11.7', 'cloudpickle': '3.1.1'}
The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'google-adk', 'cloudpickle', 'pydantic', 'python-dotenv', 'pyyaml', 'google-cloud-bigquery']
Using bucket ge-speca-sandbox-adk-deploy
Wrote to gs://ge-speca-sandbox-adk-deploy/agent_engine/agent_engine.pkl
Writing to gs://ge-speca-sandbox-adk-deploy/agent_engine/requirements.txt
Creating in-memory tarfile of extra_packages
Writing to gs://ge-speca-sandbox-adk-deploy/agent_engine/dependencies.tar.gz
Bidi stream API mode is not supported yet in Vertex SDK, please use the GenAI SDK instead. Skipping method bidi_stream_query.
Update Agent Engine backing LRO: projects/267339081837/locations/us-central1/reasoningEngines/7451852096321617920/operations/7520512800725139456
Agent Engine updated. Resource name: projects/267339081837/locations/us-central1/reasoningEngines/7451852096321617920


In [ ]:
# Deploy the Agent on AI Engine (This takes a few minutes)
# remote_agent = agent_engines.create(
#     agent_engine = adk_agent,             # The Agent instantiated as ADK agent
#     requirements=requirements,            # Requirements file
#     extra_packages=extra_packages,        # Extra packages
#     display_name=config['agent_display_name'],    # Display name  
#     description=config['agent_description'],     # Description
#     env_vars=env_vars                     # Env Vars dict
# )

### Test Remote Agent on Agent Engine

In [30]:
## Retrieve all existent Agent Engine resource.names (Agents)
# To confirm new agent was deployed
for agent in agent_engines.list():
    print(f"============================ \nAgent: {agent.display_name}\nResoruce Name: {agent.resource_name}\nCreated/updated at: {agent.update_time} \n\n" )

Agent: ADK BQ Agent OAuth
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/7451852096321617920
Created/updated at: 2025-11-21 16:09:13.935590+00:00 


Agent: Agente Recursos Humanos
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/8886459683871653888
Created/updated at: 2025-11-13 20:31:33.996791+00:00 


Agent: ADK VAIS Search Agent
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/5370837224755560448
Created/updated at: 2025-11-11 18:59:13.814593+00:00 


Agent: ADK OAuth Agent
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/1907964935493648384
Created/updated at: 2025-10-28 17:50:34.549372+00:00 




In [31]:
# Confirm that "remote_agent" is pointing to your new agent
print(f"=================== Remote Agent ============================ \n\
 Name: {remote_agent.display_name}\n\
 Resoruce Name: {remote_agent.resource_name}\n\
 Created/updated at: {remote_agent.update_time} \n\n" )

=================== Remote Agent ============================ 
 Name: ADK BQ Agent OAuth
 Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/7451852096321617920
 Created/updated at: 2025-11-21 16:09:13.935590+00:00 




In [32]:
# Run a simple query
for remote_event in remote_agent.stream_query(
    user_id="user",
    message="Hi, what you can do form me?",
):
    display(JSON(remote_event,expanded=True)) 

<IPython.core.display.JSON object>

In [33]:
# Formatted final output
display(Markdown(f"```json\n{json.dumps(remote_event, indent=2,ensure_ascii=False)}\n```"))

```json
{
  "model_version": "gemini-2.5-flash",
  "content": {
    "parts": [
      {
        "text": "I can help you by writing and executing BigQuery SQL queries to retrieve information from the Moodle database. Just tell me what you're looking for! For example, you can ask me to:\n\n*   Find information about courses, users, assignments, or grades.\n*   List users enrolled in a specific course.\n*   Calculate average grades for assignments.\n*   Identify courses with no assignments.\n*   Count courses per category.\n*   And much more!\n\nWhat kind of information are you interested in today?"
      }
    ],
    "role": "model"
  },
  "finish_reason": "STOP",
  "usage_metadata": {
    "candidates_token_count": 109,
    "candidates_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 109
      }
    ],
    "prompt_token_count": 9481,
    "prompt_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 9481
      }
    ],
    "total_token_count": 9590,
    "traffic_type": "ON_DEMAND"
  },
  "avg_logprobs": -0.10583234489510912,
  "invocation_id": "e-b0c93da6-89b5-47b8-aa12-bd64d6e65a6b",
  "author": "bigquery_agent",
  "actions": {
    "state_delta": {},
    "artifact_delta": {},
    "requested_auth_configs": {},
    "requested_tool_confirmations": {}
  },
  "id": "64d53e7b-09a5-42e3-9a7c-1eac12b29662",
  "timestamp": 1763741385.609985
}
```